In [1]:
import confnotebook

In [2]:
from pathlib import Path

source = Path("../examples/RPA-6542")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 18470938
[1] 18470982
[2] 18548791
[3] 18561292
[4] 18561867
[5] 18639563
[6] 18660877
[7] 18667876
[8] 18667914
[9] 18668755
[10] 18674893
[11] 18687424
[12] 18690095
[13] 18690959
[14] 18692621
[15] 18699963
[16] 18892339
[17] 18892958
[18] 18892969
[19] 18897098
[20] 7-1
[21] [Untitled]_23-48
[22] [Untitled]_38-48


In [3]:
IDX_FILE = 1

In [4]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
output_dir = f"../examples/output/{file.stem}"

debug_image_observer = DebugImageObserver(output_dir=output_dir)

/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [5]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline(debug_image=debug_image_observer)

document = pipeline.build(file.read_bytes())

/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/PP-OCRv5_server_det')
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/cyrillic_PP-OCRv5_mobile_rec')
2026-07-22 00:54:13.372 | INFO     | vision_core.pipelines.build_document:build:105 - Обработка страницы 0 с dpi 200...
2026-07-22 00:54:13.434 | INFO     | vision_core.pipelines.build_document:_process_page:187 - Коррекция ориентации и наклона...
2026-07-22 00:54:13.576 | DEBUG    | vision_core.preprocessor.image_orientation:process:44 - Ориентация страницы: 180

In [6]:
from app.infrastructure.services.structured_data_extractor import ReconciliationActExtractor

extractor = ReconciliationActExtractor()
data = await extractor.extract(document)

2026-07-22 00:54:17.838 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_text:70 - summary_text: 712 символов из 1 страниц
2026-07-22 00:54:17.839 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'дебет' и 'кредит' (ratio=0.33)
2026-07-22 00:54:17.839 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_cell_texts:93 - summary_cell_texts: ['По данным ООО "ПО "АЛЬФА-МЕТАЛЛ", руб.', 'Пo данным']
2026-07-22 00:54:17.839 | DEBUG    | extractor.process:extract:19 - Текст до нормализации: Акт сверки взаимных расчетов за период: 4 квартал 2025 г. между О0О "ПО "АЛЬФА-МЁТАЛЛ" (ИНН 5021011203) и O00 "КРАМ3" (ИНН 2465043748) Мы, нижеподписавшиеся,Генеральный Директор ООО "ПО "АЛьФА-МЕТАЛЛ" Трищенко Дмитрий Валентинович, с одной стороны, и , с другой стороны, составили настоящий акт сверки в том, что состояние взаимных расчетов по данным учета OOO "KPAM3" следующее: ПО ДАННЫМ ООО "ПОЧАЛЬФА-МЕ

2026-07-22 00:54:17.840 | DEBUG    | extractor.process:extract:21 - Текст после нормализации: АКТ СВЕРКИ ВЗАИМНЫХ РАСЧЕТОВ ЗА ПЕРИОД: 4 КВАРТАЛ 2025 Г. МЕЖДУ ООО "ПО "АЛЬФА-МЕТАЛЛ" (ИНН 5021011203) И O00 "КРАМ3" (ИНН 2465043748) МЫ, НИЖЕПОДПИСАВШИЕСЯ, ГЕНЕРАЛЬНЫЙ ДИРЕКТОР ООО "ПО "АЛЬФА-МЕТАЛЛ" ТРИЩЕНКО ДМИТРИЙ ВАЛЕНТИНОВИЧ, С ОДНОЙ СТОРОНЫ, И, С ДРУГОЙ СТОРОНЫ, СОСТАВИЛИ НАСТОЯЩИЙ АКТ СВЕРКИ В ТОМ, ЧТО СОСТОЯНИЕ ВЗАИМНЫХ РАСЧЕТОВ ПО ДАННЫМ УЧЕТА OOO "KPAM3" СЛЕДУЮЩЕЕ: ПО ДАННЫМ ООО "ПОЧАЛЬФА-МЕТАЛЛ" НА 31.12.2025 ЗАДОЛЖЕННОСТЬ В ПОЛЬЗУ ООО "ПО "АЛЬФА-МЕТАЛЛ" 34 670 069,04 РУБ. ТРИДЦАТЬ ЧЕТЫРЕ МИЛЛИОНА ШЕСТЬСОТ СЕМЬДЕСЯТ ТЫСЯЧ ШЕСТЬДЕСЯТ ДЕВЯТЬ РУБЛЕЙ 04 КОПЕЙКИ) ООО "ПО "АЛЬФА-МЕТАЛЛ СТАРШИЯ ЕУХГАЛТЕР ЕНЕРАЛЬНЫЙ ДИРЕКТОР ЕРМАКОВАО. Т. ДО(ТРИЩЕНКО Д. В.)23.01.24 GT 23.01.24. ПО ДАННЫМ ООО "КРАМЗ"
2026-07-22 00:54:17.841 | DEBUG    | extractor.process:extract:23 - Извлечённые токены: [DateReference(token=Token(start=40, end=54, text='4 КВАРТАЛ 2025'), date='01.10.2025', date_end='31.12.

DcExtractionError: Ошибка извлечения дебет/кредит из таблицы '0': колонки 'seller' не найдены

In [ ]:
print(data.debit)

In [ ]:
from app.application.dto.fill_reconciliation_act import FillReconciliationActCommand
from app.domain.entities.process import ProcessState
from app.infrastructure.services.pdf_filler import DocumentPdfFiller

process_state = ProcessState(
    process_id="notebook-test",
    source_pdf=files[IDX_FILE].read_bytes(),
    document_payload=document,
)

comments = f"""
            По данным покупателя {data.buyer}
            По данным продавца {data.seller}
            В период: {data.period.start} - {data.period.end}
            """

# используем значения продавца для заполнения колонок покупателя
command = FillReconciliationActCommand(
    process_id="notebook-test",
    comments=comments,
    debit=data.debit,
    credit=data.credit,
)

filler = DocumentPdfFiller()
filled_pdf = await filler.fill(process_state, command)

In [ ]:
out_path = f"../examples/output/{files[IDX_FILE].stem}_filled.pdf"
Path(out_path).write_bytes(filled_pdf)
print(out_path)